In [26]:
import os
import time,json
import ffmpeg
import random
import subprocess

from appium import webdriver as appium_webdriver
from appium.webdriver.common.appiumby import AppiumBy
from appium.options.android import UiAutomator2Options

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from seleniumwire import webdriver as wire_webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from appium.webdriver.extensions.android.nativekey import AndroidKey
from selenium.webdriver.common.actions.pointer_input import PointerInput
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse, quote_plus, unquote
from dotenv import load_dotenv

# Load .env file relative to this config.py
current_path = os.getcwd()
ENV_PATH = os.path.join(current_path, ".env")
print(ENV_PATH)
load_dotenv(dotenv_path=ENV_PATH, override=True)

# --- Environment Configuration ---
ENV = os.getenv("ENV", "production")
PORT = int(os.getenv("PORT", 5001))
WS_SCRCPY_PORT = int(os.getenv("WS_SCRCPY_PORT", 5001))
ADMIN_KEY = os.getenv("BACKEND_ADMIN_KEY")
TOKEN_EXPIRE_LIMIT = int(os.getenv("ADMIN_TOKEN_EXPIRE_LIMIT", 1))
DEV_API_KEY = os.getenv("BACKEND_DEV_API_KEY")
DEVICES_RANGE = os.getenv("DEVICES_RANGE", "192.168.1.0/24")
SECRET_KEY = os.getenv("FLASK_SECRET_KEY", "a_default_secret_key_if_not_set")

# --- Database (Postgres) ---
SQLALCHEMY_DATABASE_URI = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)
SQLALCHEMY_TRACK_MODIFICATIONS = False

# --- CORS Configuration ---
CORS_ORIGINS = os.getenv("CORS_ORIGINS", "").split(",")

# --- Clerk Authentication ---
CLERK_SECRET_KEY = os.getenv("CLERK_SECRET_KEY")
CLERK_ISSUER = os.getenv("CLERK_ISSUER")
CLERK_JWKS_URL = os.getenv("CLERK_JWKS_URL")

# --- Paths ---
BIN_FOLDER = os.path.join(current_path, "bin")
JSON_FOLDER = os.path.join(BIN_FOLDER, "files")
DATABASE_DIR = os.path.join(BIN_FOLDER, "database")
WS_SCRCPY_PATH = os.path.join(current_path, "ws-scrcpy")
ADB_SCRIPTS_PATH = os.path.join(current_path, "scripts")
FILE_UPLOAD_FOLDER = os.path.join(current_path, "uploads")

# Ensure directories exist
for folder in [BIN_FOLDER, JSON_FOLDER, DATABASE_DIR, FILE_UPLOAD_FOLDER]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# --- JSON Databases ---
FILES_JSON_PATH = os.path.join(JSON_FOLDER, "files.json")
if os.path.exists(FILES_JSON_PATH):
    with open(FILES_JSON_PATH, "r") as f:
        UPLOADED_FILES_DATABASE = json.load(f)
else:
    UPLOADED_FILES_DATABASE = {}

APPIUM_CONTAINERS_DATABASE = os.path.join(JSON_FOLDER, "running_containers.json")

# --- Server Configuration ---
FLASK_RUN_PORT = int(os.getenv("BACKEND_PORT", 5000))

# --- Cloudflare Tunnels ---
CLOUDFLARED_TUNNEL_TOKEN = os.getenv("CLOUDFLARED_TUNNEL_TOKEN")

# --- Apps Package & Activities ---
YT_MUSIC_PACKAGE_NAME = os.getenv('YT_MUSIC_PACKAGE_NAME')
YT_MUSIC_ACTIVITY_NAME = os.getenv('YT_MUSIC_ACTIVITY_NAME')


# Skipping Rates
SKIPPING_RATES_COUNT = 8
SKIPPING_HIGH_COUNT  = 1
SKIPPING_LOW_RANGE   = [42,  58]
SKIPPING_HIGH_RANGE  = [100, 120]
SKIPPING_AVG_RANGE   = [55,  60]


device_id = "192.168.1.55:5555"

PACKAGE_ID = "com.google.android.apps.youtube.music" # "com.google.android.gms"
ACTIVITY = "com.google.android.apps.youtube.music.activities.MusicActivity" # 'com.google.android.gms/com.google.android.gms.auth.uiflows.minutemaid.MinuteMaidActivity' #
APPIUM_SERVER = "http://127.0.0.1:4723" 

/home/magician/Desktop/MagSpot/notebook/.env


In [2]:
def is_app_running(device_id, package_name):
    result = subprocess.getoutput(f"adb -s {device_id} shell pidof {package_name}")
    return bool(result.strip())

def create_driver(device_id, PACKAGE_ID, ACTIVITY):
    options = UiAutomator2Options().load_capabilities({
        "platformName": "Android",
        "automationName": "UiAutomator2",
        "udid": device_id,
        "deviceName": device_id,

        "appPackage": PACKAGE_ID,
        "appActivity": ACTIVITY,

        "autoLaunch": True,
        "forceAppLaunch": True,
        "noReset": True,

        "newCommandTimeout": 3600,
    })
    driver = appium_webdriver.Remote(command_executor=APPIUM_SERVER, options=options)
    if not is_app_running(device_id, PACKAGE_ID): driver.start_activity(PACKAGE_ID, ACTIVITY)
    time.sleep(4)
    return driver


def scroll_down(driver):    
    # get device screen size
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    # convert percentages to pixels
    start_x = int(width * 0.50)   # 50%
    start_y = int(height * 0.70)  # 70%
    end_x   = int(width * 0.50)   # 50%
    end_y   = int(height * 0.30)  # 30%
    # build gesture
    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(start_x, start_y)  # move to start
    actions.pointer_action.pointer_down()                      # finger down
    actions.pointer_action.move_to_location(end_x, end_y)  # swipe
    actions.pointer_action.pointer_up()                        # finger up
    # perform action
    actions.perform()

In [3]:
import time
import random
import subprocess
import os, ffmpeg
from appium import webdriver as appium_webdriver
from appium.webdriver.common.appiumby import AppiumBy
from appium.options.android import UiAutomator2Options

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from seleniumwire import webdriver as wire_webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from appium.webdriver.extensions.android.nativekey import AndroidKey
from selenium.webdriver.common.actions.pointer_input import PointerInput
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse, quote_plus, unquote


def tapOnScreenCoord(driver: appium_webdriver, x: int, y: int):
    """
    Tap on the screen at absolute coordinates (x, y).
    """
    driver.tap([(x, y)])
    time.sleep(1)


def tapOnScreen(driver: appium_webdriver, x_per=0.93, y_per=0.29):
    # get screen size
    size = driver.get_window_size()
    print(size)
    width = size['width']
    height = size['height']
    # convert percentages to pixels
    x = int(width * x_per)
    y = int(height * y_per)
    driver.tap([(x, y)])
    time.sleep(1)


def setBackgrounMusicVolume(driver, x_per=0.17, y_per=0.89):
    # Get screen size
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    # Convert percentages → pixels
    x = int(width * x_per)   # 17%
    y = int(height * y_per)  # 89%
    # Build tap gesture
    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(x, y)
    actions.pointer_action.pointer_down()
    actions.pointer_action.pointer_up()
    actions.perform()
    time.sleep(1)


import random
import time
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.actions.pointer_input import PointerInput

def scroll_up(driver, distance_perc: float = 0.5, duration: int = 800):
    """
    Scroll up like a human.
    :param distance_perc: fraction of screen height to scroll (default 0.5 = 50%)
    :param duration: swipe duration in ms (default 800ms ~ human speed)
    """
    size = driver.get_window_size()
    width, height = size['width'], size['height']

    start_x = int(width * 0.5)
    start_y = int(height * (0.7))   # start lower
    end_x   = start_x
    end_y   = int(start_y - height * distance_perc)

    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(start_x, start_y)
    actions.pointer_action.pointer_down()
    actions.pointer_action.move_to_location(end_x, end_y, duration=duration)
    actions.pointer_action.pointer_up()
    actions.perform()


def scroll_down(driver, distance_perc: float = 0.5, duration: int = 800):
    """
    Scroll down like a human.
    :param distance_perc: fraction of screen height to scroll (default 0.5 = 50%)
    :param duration: swipe duration in ms (default 800ms ~ human speed)
    """
    size = driver.get_window_size()
    width, height = size['width'], size['height']

    start_x = int(width * 0.5)
    start_y = int(height * (0.3))   # start higher
    end_x   = start_x
    end_y   = int(start_y + height * distance_perc)

    actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
    actions.pointer_action.move_to_location(start_x, start_y)
    actions.pointer_action.pointer_down()
    actions.pointer_action.move_to_location(end_x, end_y, duration=duration)
    actions.pointer_action.pointer_up()
    actions.perform()



def lower_video_quality(input_file, output_dir="compressed_videos", qualities=None):
    """
    Create lower-quality versions of a video.
    Args:
        input_file (str): Path to the input video.
        output_dir (str): Directory to save converted videos.
        qualities (dict): Mapping of resolution name -> scale height (width auto).
        Example: {"720p": 720, "480p": 480, "360p": 360}
    """
    if qualities is None: qualities = {"720p": 720, "480p": 480, "360p": 360}
    if not os.path.exists(output_dir): os.makedirs(output_dir)

    base_name = os.path.splitext(os.path.basename(input_file))[0]
    output_files = {}

    for label, height in qualities.items():
        out_path = os.path.join(output_dir, f"{base_name}_{label}.mp4")
        try:
            (
                ffmpeg
                .input(input_file)
                .output(out_path, vf=f"scale=-2:{height}", vcodec="libx264", crf=28, preset="fast")
                .overwrite_output()
                .run(quiet=True)
            )
            output_files[label] = out_path
            # print(f"✅ Saved {label} video: {out_path}")
        except Exception as e: print(f"❌ Failed to create {label} version: {e}")
    return output_files



def push_video_to_android(video_path, device_id):
    try:
        # STEP 1: Push Video To Android Phone
        local_file = video_path
        # pushed_file = os.path.join(FILE_UPLOAD_FOLDER, f"Video_{random.randint(1111,9999)}.mp4")
        # output_ = lower_video_quality(local_file, FILE_UPLOAD_FOLDER, qualities = {"720p": 720})
        output_ = {'720p': local_file}
        remote_path = f"/storage/emulated/0/Download/{os.path.basename(output_['720p'])}"

        # Push file
        subprocess.run(["adb", "-s", device_id, "push", output_['720p'], remote_path])

        # Trigger MediaScanner
        subprocess.run([
            "adb", "-s", device_id, "shell", 
            "am", "broadcast", "-a", "android.intent.action.MEDIA_SCANNER_SCAN_FILE", 
            "-d", f"file://{remote_path}"
        ])
        return remote_path, output_
    except Exception as e:
        raise Exception(f"ERROR: Failed To Push Video To Android {device_id} PATH:{video_path}. \n{e} --Output: {output_}")



def remove_video_from_android(android_video, device_id):
    subprocess.run([
        "adb", "-s", device_id, "shell", 
        "rm", f"/storage/emulated/0/Download/{os.path.basename(android_video['720p'])}"
    ])



def get_screen_resolution(driver):
    """
    Get the current screen resolution from an Appium driver.
    Returns (width, height) as integers.
    """
    size = driver.get_window_size()
    width = size['width']
    height = size['height']
    return width, height




In [35]:

# MAIN BUTTONS
HOME_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Home"]'
HOME_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Home']"

SAMPLES_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Samples"]'
SAMPLES_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Samples']"

EXPLORE_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Explore"]'
EXPLORE_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Explore']"

LIBRARY_BUTTON_XPATH1 = '//android.widget.TextView[@resource-id="com.google.android.apps.youtube.music:id/text1" and @text="Library"]'
LIBRARY_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/pivot_bar']//*[@text='Library']"

# SEARCH BUTTON
SEARCH_BUTTON_XPATH1 = '//android.widget.ImageButton[@content-desc="Search"]'
SEARCH_BUTTON_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/action_search_button']"

# SEARCH INPUT
SEARCH_INPUT_XPATH1 = '//android.widget.EditText[@resource-id="com.google.android.apps.youtube.music:id/search_edit_text"]'
SEARCH_INPUT_XPATH2 = "//*[@resource-id='com.google.android.apps.youtube.music:id/search_edit_text']"

# SEARCH_DIRECTOR
YT_MUSIC_SEARCH_DIRECTORY_XPATH = '//android.widget.LinearLayout[@content-desc="YT Music"]'
LIBRARY_SEARCH_DIRECTORY_XPATH = '//android.widget.LinearLayout[@content-desc="Library"]'

# SEARCH FILTERS
SEARCH_FILTERS_BOX_XPATH1  = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']"
SEARCH_FILTERS_BOX_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]'

SEARCH_FILTERS_ARTISTS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Artists']" 
SEARCH_FILTERS_ARTISTS_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]//*[@text="Artists"]'

SEARCH_FILTERS_ARTISTS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Artists']" 
SEARCH_FILTERS_ARTISTS_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]//*[@text="Artists"]'

SEARCH_FILTERS_PROFILES_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Profiles']" 
SEARCH_FILTERS_PROFILES_XPATH2 = '//android.support.v7.widget.RecyclerView[@resource-id="com.google.android.apps.youtube.music:id/chip_cloud"]//*[@text="Profiles"]'

SEARCH_FILTERS_ALBUMS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Albums']" 
SEARCH_FILTERS_ALBUMS_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Albums']"

SEARCH_FILTERS_SONGS_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Songs']" 
SEARCH_FILTERS_SONGS_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Songs']"

SEARCH_FILTERS_COMMUNITY_PL_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Community playlists']" 
SEARCH_FILTERS_COMMUNITY_PL_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Community playlists']"

SEARCH_FILTERS_FEATURED_PL_XPATH1 = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Featured playlists']" 
SEARCH_FILTERS_FEATURED_PL_XPATH2 = "//android.support.v7.widget.RecyclerView[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']//*[@text='Featured playlists']"

SEARCH_FILTER_CUSTOM_XPATH = lambda filter_name: f"""//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='{filter_name}']"""

# SEARCH_GO_BACK
SEARCH_GO_BACK_BUTTON_XPATH = '//android.widget.ImageView[@content-desc="Search back"]'

# MUSIC GO BACK
MUSIC_GO_BACK_BUTTON_XPATH = '//android.widget.ImageButton[@content-desc="Back"]'


# SEARH RESULTS
SEARCH_RESULTS_BOX_XPATH = '//*[@resource-id="com.google.android.apps.youtube.music:id/results_list"]'
SELECT_TARGET_FROM_RESULTS = lambda matching_text: f"""//*[@resource-id='com.google.android.apps.youtube.music:id/results_list']
  //*[contains(
      translate(@text,'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),
      '{matching_text.lower()}'
  )]
"""
## For Clicking 3 DOTS We Need To First Find The Localtion Of Target Div And TargetCoordinate
## rect gives x, y, width, height 
# rect = element.rect x, y, width, height = rect["x"], rect["y"], rect["width"], rect["height"] 
## endpoint coordinates (bottom-right corner) end_x = x + width end_y = y + 

SHUFFLE_PLAY_BUTTON_XPATH1 = '//android.view.ViewGroup[@content-desc="Shuffle play"]'
SHUFFLE_PLAY_BUTTON_XPATH2 = '//*[@resource-id="com.google.android.apps.youtube.music:id/results_list"]//*[contains(@content-desc, "Shuffle")]'
START_MIX_BUTTON_XPATH = '//android.view.ViewGroup[@content-desc="Start mix"]'
CLOSE_MUSIC_PROPERTIES_XPATH = '//android.view.ViewGroup[@content-desc="Close"]'

MUSIC_DOWN_BUTTON_XPATH = "//*[@resource-id='com.google.android.apps.youtube.music:id/toolbar_back_navigation']"

POPUP_RUNNING_MUSIC='//*[@resource-id="com.google.android.apps.youtube.music:id/mini_player"]'


LIBRARY_FILTER_BOX_XPATH = "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud']"
LIBRARY_FILTER_PLAYLISTS_XPATH =  "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Playlists']"
LIBRARY_FILTER_SONGS_XPATH =  "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Songs']"
LIBRARY_FILTER_ARTISTS_XPATH =  "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Artists']"
LIBRARY_FILTER_ALBUMS_XPATH =  "//*[@resource-id='com.google.android.apps.youtube.music:id/chip_cloud_chip_text' and @text='Albums']"
LIBRARY_SEARCH_PLAYLIST = lambda playlist_name: f"""//*[
    @resource-id="com.google.android.apps.youtube.music:id/title"
    and (
        contains(translate(@text,'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), "{playlist_name.lower()}")
        or
        contains(translate(@content-desc,'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), "{playlist_name.lower()}")
    )
]"""
LIBRARY_PLAYLIST_PLAY_BUTTON_XPATH = '//*[@content-desc="Play"]'

In [5]:
import random
import math



def generate_skipping_rates(
    count: int = 8,
    low_range: list = [42, 58],
    high_range: list = [100, 120],
    high_count: int = 1,
    avg_range: list = [55, 60]
):
    """
    Generate 'count' numbers with constraints:
    - Most numbers from low_range
    - 'high_count' numbers from high_range
    - Average must fall within avg_range
    """

    while True:
        # Step 1: Pick random indices for high numbers
        high_indices = random.sample(range(count), high_count)

        # Step 2: Generate numbers in low range
        nums = random.sample(range(low_range[0], low_range[1]), count)

        # Step 3: Replace selected positions with high range numbers
        for idx in high_indices:
            nums[idx] = random.randint(high_range[0], high_range[1])

        # Step 4: Check average constraint
        avg = sum(nums) / count
        if avg_range[0] <= avg <= avg_range[1]:
            return nums, avg



BASE_WIDTH = 1080
BASE_HEIGHT = 2280





def random_point_in_rectangle(left, top, right, bottom, device_width, device_height):
    """
    Generate random point inside rectangle defined by (left, top) and (right, bottom).
    Scales coordinates from baseline (1080x2280) to current device resolution.
    """
    # shrink boundaries by 1% to stay inside
    width = right - left
    height = bottom - top
    shrink_x = width * 0.01
    shrink_y = height * 0.01

    # random point inside shrunken rectangle
    x = random.uniform(left + shrink_x, right - shrink_x)
    y = random.uniform(top + shrink_y, bottom - shrink_y)

    # scale to current resolution
    scaled_x = int(x / BASE_WIDTH * device_width)
    scaled_y = int(y / BASE_HEIGHT * device_height)

    return scaled_x, scaled_y

    
BUTTON_CENTERS = {
    "shuffle": {
        "base":    (75, 1760),
        "physical": (75, 1270),
        "radius": 60
    },
    "loop": {
        "base":    (990, 1760),
        "physical": (645, 1270),
        "radius": 60
    },
    "next": {
        "base":    (785, 1760),
        "physical": (520, 1270),
        "radius": 70
    },
    "back": {
        "base":    (280, 1760),
        "radius": 70
    },
    "play": {
        "base":    (535, 1770),
        "physical": (355, 1270),
        "radius": 90
    },
    "like": {
        "base":    (130, 1465),
        "physical": (115, 1050),
        "radius": 40
    },
    "dislike": {
        "base":    (290, 1465),
        "physical": (214, 1050),
        "radius": 40
    }
}


def random_point_in_circle(center_x, center_y, radius, device_width, device_height, isOverrideResolution=True):
    """
    Generate a random point inside a circle, scaled to device resolution.
    
    :param isOverrideResolution: True = use BASE coordinates scaling, False = use physical coordinates
    """
    # Scale center coordinates from BASE to device
    if isOverrideResolution:
        center_x = center_x / BASE_WIDTH * device_width
        center_y = center_y / BASE_HEIGHT * device_height
    # If physical device, center is already in physical coordinates
    # but we still need to scale radius relative to device size
    scale_factor = min(device_width / BASE_WIDTH, device_height / BASE_HEIGHT)
    radius *= scale_factor * 0.99  # shrink slightly for safety

    r = radius * math.sqrt(random.random())
    theta = random.uniform(0, 2 * math.pi)

    x = center_x + r * math.cos(theta)
    y = center_y + r * math.sin(theta)

    return int(x), int(y)





def _get_btn_coord(name, device_width, device_height, isOverrideResolution=False):
    if name not in BUTTON_CENTERS:
        raise ValueError(f"Button '{name}' not found in BUTTON_CENTERS")

    center_key = "base" if isOverrideResolution else "physical"
    center_x, center_y = BUTTON_CENTERS[name][center_key]
    radius = BUTTON_CENTERS[name]["radius"]
    print(name, center_x, center_y, device_width, device_height, isOverrideResolution)
    return random_point_in_circle(center_x, center_y, radius, device_width, device_height, isOverrideResolution)


def get_shuffle_btn_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("shuffle", device_width, device_height, isOverrideResolution)

def get_loop_btn_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("loop", device_width, device_height, isOverrideResolution)

def get_next_btn_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("next", device_width, device_height, isOverrideResolution)

def get_back_btn_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("back", device_width, device_height, isOverrideResolution)

def get_play_btn_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("play", device_width, device_height, isOverrideResolution)

def get_like1_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("like", device_width, device_height, isOverrideResolution)

def get_dislike_btn_coord(device_width, device_height, isOverrideResolution=False):
    return _get_btn_coord("dislike", device_width, device_height, isOverrideResolution)


def get_random_click_coord(device_width, device_height):
    # Rectangle defined by your coordinates
    left, top = 660, 1250
    right, bottom = 1000, 1350
    return random_point_in_rectangle(left, top, right, bottom, device_width, device_height)



if __name__ == "__main__":
    numbers, average = generate_skipping_rates()
    print("Numbers:", numbers)
    print("Average:", average)


Numbers: [55, 57, 44, 49, 50, 43, 101, 48]
Average: 55.875


In [6]:
class ScrollLimitReached(Exception):
    """Raised when no more scrolling is possible."""
    pass

import hashlib
import time

def _page_fingerprint(driver):
    return hashlib.md5(driver.page_source.encode("utf-8")).hexdigest()

from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.actions.pointer_input import PointerInput

def scroll_up(driver, percent=40):
    size = driver.get_window_size()
    width = size['width']
    height = size['height']

    scroll_px = height * (percent / 100)

    start_x = width // 2
    start_y = int((height / 2) - (scroll_px / 2))
    end_y   = int((height / 2) + (scroll_px / 2))

    before = _page_fingerprint(driver)
    try:
        actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
        finger = actions.pointer_action

        finger.move_to_location(start_x, start_y)
        finger.pointer_down()
        finger.move_to_location(start_x, end_y)
        finger.pointer_up()
        actions.perform()
        time.sleep(0.4)
        after = _page_fingerprint(driver)
        if before == after:
            raise ScrollLimitReached("Reached TOP – no more content to scroll up")

    except ScrollLimitReached:  raise Exception("SCROLL LIMIT: Scrolled To The Top Of Screen")
    except Exception as e: raise RuntimeError(f"Scroll up failed: {e}")

    
def scroll_down(driver, percent=40):
    size = driver.get_window_size()
    width = size['width']
    height = size['height']

    scroll_px = height * (percent / 100)

    start_x = width // 2
    start_y = int((height / 2) + (scroll_px / 2))
    end_y   = int((height / 2) - (scroll_px / 2))

    before = _page_fingerprint(driver)

    try:
        actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
        finger = actions.pointer_action

        finger.move_to_location(start_x, start_y)
        finger.pointer_down()
        finger.move_to_location(start_x, end_y)
        finger.pointer_up()
        actions.perform()
        time.sleep(0.4)
        after = _page_fingerprint(driver)
        if before == after:
            raise ScrollLimitReached("Reached BOTTOM – no more content to scroll down")

    except ScrollLimitReached: raise Exception("SCROLL LIMIT: Scrolled To The Bottom Of Screen")
    except Exception as e: raise RuntimeError(f"Scroll down failed: {e}")





def tap_random_in_element(driver: webdriver, element):
    """
    Tap at a random point inside the given element.
    The point is chosen uniformly within a circle inscribed in the element bounds.
    """
    # get element location and size
    loc = element.location
    size = element.size

    center_x = loc['x'] + size['width'] / 2
    center_y = loc['y'] + size['height'] / 2

    # radius is half of the smaller dimension (to stay inside element)
    radius = min(size['width'], size['height']) / 2

    # random polar coordinates inside circle
    r = radius * math.sqrt(random.random())  # uniform distribution
    theta = random.uniform(0, 2 * math.pi)

    x = int(center_x + r * math.cos(theta))
    y = int(center_y + r * math.sin(theta))

    driver.tap([(x, y)])
    time.sleep(0.5)  # small delay

    return (x, y)  # return coords for debugging/logging

In [39]:
import time
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.actions.pointer_input import PointerInput

def _get_scroll_center(driver, element=None):
    """Return center coordinates of window or element if provided."""
    if element:
        rect = element.rect
        center_x = rect['x'] + rect['width'] // 2
        center_y = rect['y'] + rect['height'] // 2
    else:
        size = driver.get_window_size()
        center_x = size['width'] // 2
        center_y = size['height'] // 2
    return center_x, center_y

def left_scroll(driver, percent=40, element=None):
    size = driver.get_window_size()
    width = size['width']

    scroll_px = width * (percent / 100)
    center_x, center_y = _get_scroll_center(driver, element)

    start_x = int(center_x + (scroll_px / 2))
    end_x   = int(center_x - (scroll_px / 2))
    start_y = center_y

    before = _page_fingerprint(driver)
    try:
        actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
        finger = actions.pointer_action

        finger.move_to_location(start_x, start_y)
        finger.pointer_down()
        finger.move_to_location(end_x, start_y)
        finger.pointer_up()
        actions.perform()
        time.sleep(0.4)

        after = _page_fingerprint(driver)
        if before == after:
            raise ScrollLimitReached("Reached LEFT – no more content to scroll left")

    except ScrollLimitReached:
        raise Exception("SCROLL LIMIT: Scrolled To The Left Edge Of Screen")
    except Exception as e:
        raise RuntimeError(f"Left scroll failed: {e}")


def right_scroll(driver, percent=40, element=None):
    size = driver.get_window_size()
    width = size['width']

    scroll_px = width * (percent / 100)
    center_x, center_y = _get_scroll_center(driver, element)

    start_x = int(center_x - (scroll_px / 2))
    end_x   = int(center_x + (scroll_px / 2))
    start_y = center_y

    before = _page_fingerprint(driver)
    try:
        actions = ActionBuilder(driver, mouse=PointerInput("touch", "finger"))
        finger = actions.pointer_action

        finger.move_to_location(start_x, start_y)
        finger.pointer_down()
        finger.move_to_location(end_x, start_y)
        finger.pointer_up()
        actions.perform()
        time.sleep(0.4)

        after = _page_fingerprint(driver)
        if before == after:
            raise ScrollLimitReached("Reached RIGHT – no more content to scroll right")

    except ScrollLimitReached:
        raise Exception("SCROLL LIMIT: Scrolled To The Right Edge Of Screen")
    except Exception as e:
        raise RuntimeError(f"Right scroll failed: {e}")


ACTION_MENU_BUTTON_XPAHT1 = """//*[@resource-id="com.google.android.apps.youtube.music:id/elements_container"]//*[@content-desc="Action menu"]"""
ACTIION_MENU_SHUFFLE_PLAY_BUTTON = """//*[@resource-id="com.google.android.apps.youtube.music:id/text" and @text="Shuffle play"]"""

In [27]:
driver = create_driver(device_id, PACKAGE_ID, ACTIVITY)

In [28]:
# STEP 1: Click Home Tab...
print("--> Clicking Home Tab...")
try: WebDriverWait(driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, HOME_BUTTON_XPATH1))).click()
except: 
    try: WebDriverWait(driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, HOME_BUTTON_XPATH2))).click()
    except Exception as e:  raise Exception(f"Failed to Click Home Tab... \n{e}")

--> Clicking Home Tab...


In [ ]:
# --> Clicking Search Icon..
print("--> Clicking Search Icon...")
try: WebDriverWait(driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SEARCH_BUTTON_XPATH1))).click()
except: 
    try: WebDriverWait(driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SEARCH_BUTTON_XPATH2))).click()
    except Exception as e: raise Exception(f"Failed to Click Search Icon... \n{e}")

--> Clicking Search Icon...


In [ ]:
# --> Clicking Playlist Filter From Search Filter...
print("--> Clicking Playlist Filter From Search Filter...")
search_inp_obj = None
playlist_name = "naz' Favorites"
try: 
    WebDriverWait(driver, timeout=10).until(EC.presence_of_element_located((By.XPATH, SEARCH_INPUT_XPATH1)))
    search_inp_obj = driver.find_element(By.XPATH, SEARCH_INPUT_XPATH1)
except: 
    try: 
        WebDriverWait(driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, SEARCH_INPUT_XPATH2)))
        search_inp_obj = driver.find_element(By.XPATH, SEARCH_INPUT_XPATH2)
    except Exception as e: raise Exception(f"Failed to Find Search Input Box... \n{e}")

try: 
    if search_inp_obj: search_inp_obj.click()
    else: raise Exception("Search Input Box Not Found...")
    search_inp_obj.send_keys(playlist_name)
    driver.execute_script("mobile: performEditorAction", {
        "action": "search"
    })
except Exception as e: raise Exception(f"Failed to Type Artist Name... \n{e}")

--> Clicking Playlist Filter From Search Filter...


In [31]:
while True:
    print("--> Selecting 'Community Playlist' from filters......")
    try:
        # WebDriverWait(driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, SEARCH_FILTER_CUSTOM_XPATH('Profiles'))))
        WebDriverWait(driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, SEARCH_FILTERS_COMMUNITY_PL_XPATH1))).click()
        break
    except: 
        filter_box = driver.find_element(By.XPATH, SEARCH_FILTERS_BOX_XPATH1)
        left_scroll(driver, percent=20, element=filter_box)

--> Selecting 'Community Playlist' from filters......


In [36]:
print("--> Finding Playlist...")
while True: 
    try: 
        WebDriverWait(driver, timeout=5).until(EC.presence_of_element_located((By.XPATH, LIBRARY_SEARCH_PLAYLIST(playlist_name))))
        print(f"--> Clicking Playlist[{playlist_name}]...")
        targetPlayList = driver.find_element(By.XPATH, LIBRARY_SEARCH_PLAYLIST(playlist_name)).click()
        break
    except Exception as e:  
        print(e)
        break
        try: scroll_down(driver, percent=25)
        except: raise

--> Finding Playlist...
--> Clicking Playlist[naz' Favorites]...


In [41]:
driver.find_element(By.XPATH, ACTION_MENU_BUTTON_XPAHT1).click()
time.sleep(4)
driver.find_element(By.XPATH, ACTIION_MENU_SHUFFLE_PLAY_BUTTON).click()

In [121]:
LIBRARY_PLAYLIST_PLAY_BUTTON_XPATH = '//*[@content-desc="Play"]'
play_button = driver.find_element(By.XPATH, LIBRARY_PLAYLIST_PLAY_BUTTON_XPATH)

coords = tap_random_in_element(driver, play_button)
print(f"Tapped at {coords}")

Tapped at (485, 1240)


In [42]:
# STEP 5: Clicking 'Shuffle' + Reat Button...
try:
    screenWidth, screenHeight = get_screen_resolution(driver)

    time.sleep(8)
    # shuffle_btn_x, shuffle_btn_y = get_shuffle_btn_coord(screenWidth, screenHeight, isOverrideResolution=True)
    # print(f"--> Clicking 'Shuffle' button [{shuffle_btn_x}, {shuffle_btn_y}]...") 
    # tapOnScreenCoord(driver, shuffle_btn_x, shuffle_btn_y)
    # time.sleep(5)

    loop_btn_x, loop_btn_y = get_loop_btn_coord(screenWidth, screenHeight, isOverrideResolution=True)
    print(f"--> Clicking 'Loop' button [{loop_btn_x}, {loop_btn_y}]...") 
    tapOnScreenCoord(driver, loop_btn_x, loop_btn_y)
    time.sleep(5)

except Exception as e:
    print(e)
    raise Exception(str(e)) 


print("--> Skipping First Song...") 
next_btn_x, next_btn_y = get_next_btn_coord(screenWidth, screenHeight, True)
tapOnScreenCoord(driver, next_btn_x, next_btn_y)
time.sleep(1.5)

loop 990 1760 1080 2280 True
--> Clicking 'Loop' button [977, 1795]...
--> Skipping First Song...
next 785 1760 1080 2280 True


In [ ]:

# STEP 4: Generating Rates And Playing Musics...
sets = 1

while True:
    
    random_skipping_rates, rates_avg = generate_skipping_rates(
        SKIPPING_RATES_COUNT,
        SKIPPING_LOW_RANGE,
        SKIPPING_HIGH_RANGE,
        SKIPPING_HIGH_COUNT,
        SKIPPING_AVG_RANGE
    )
    print(f"--> Generated Skipping Rates: [{random_skipping_rates}] Average: [{rates_avg}] Sets: [{sets}]...")
    
    for rate_index, skip_time in enumerate(random_skipping_rates):
        start = time.monotonic()

        next_btn_x, next_btn_y = get_next_btn_coord(screenWidth, screenHeight)
        tapOnScreenCoord(driver, next_btn_x, next_btn_y)
        time.sleep(1.5)

        while time.monotonic() - start < skip_time:
            time.sleep(0.5)  # check every half second
        
        print(f"--> --> [{rate_index+1}] Listened Song: [{skip_time}] Seconds")

        
    sets += 1
    break

In [ ]:
driver.quit()